# Urban vs Non-Urban Dummy Variable Creation


In [3]:
import pandas as pd
from scipy.spatial import cKDTree
import numpy as np
import h5py

In [63]:
cur_data = pd.read_csv("../../data/Fall 2024 data/cleaned_alldata_version2.csv")
pop_data = pd.read_csv("../../data/spring_2025_data/india-population-data.csv")

We will be adding a `pop_density` variable to the fa24 data by joining observations with the observation in the population data which minimizes the Euclidian distance between `(Lat, Long)` and `(CENTROID_X, CENTROID_Y)`.


In [64]:
def join_euclidean(df1, df2, df1_xy_cols, df2_xy_cols, data_col):
    """performs a join on minimum euclidean distance"""
    tree = cKDTree(df1[df1_xy_cols].values)  # Using a CKD Tree to improve efficiency
    distances, indices = tree.query(df2[df2_xy_cols].values)
    nearest = df1.iloc[indices][data_col].reset_index(drop=True)

    return pd.concat([df2.reset_index(drop=True), nearest], axis=1)


In [65]:
pop_density_joined = join_euclidean(
    pop_data, cur_data, ["CENTROID_X", "CENTROID_Y"], ["Long", "Lat"], "UN_2020_DS"
)

The Indian Ministry of Housing and Urban Affairs defines an urban area as an area that satisfies the following requirements:

(i) a minimum population of 5,000

(ii) at least 75% of male working population engaged in non-agricultural pursuits; and

(iii) a density of population of at least 400 persons per square kilometer.

[source](https://mohua.gov.in/pdf/5c80e2225a124Handbook%20of%20Urban%20Statistics%202019.pdf#page=26)

As we only have access to population density data, we will be relaxing this definition and focusing on the third (iii) requirement.


In [ ]:
pop_density_joined["is_urban"] = (pop_density_joined["UN_2020_DS"] >= 400).astype(int)
pop_density_joined["pop_bin"] = pd.cut(
    pop_density_joined["UN_2020_DS"], bins=10, labels=[x for x in range(10)]
)

### Nighttime Lights Dataset


In [67]:
TILE_SIZE = 10
PIXELS_PER_TILE = 2400
DEGREES_PER_PIXEL = TILE_SIZE / PIXELS_PER_TILE


def get_rad_df(file_path, tile_h, tile_v):
    """Extracts radiance with lat/long coordinates from night lights HDF5 file"""
    with h5py.File(file_path, "r") as f:
        radiance = f[
            "HDFEOS/GRIDS/VNP_Grid_DNB/Data Fields/Gap_Filled_DNB_BRDF-Corrected_NTL"
        ][()]

    x_coords = np.linspace(
        -180 + tile_h * TILE_SIZE,
        -180 + (tile_h + 1) * TILE_SIZE,
        PIXELS_PER_TILE,
        endpoint=False,
    )

    y_coords = np.linspace(
        90 - tile_v * TILE_SIZE,
        90 - (tile_v + 1) * TILE_SIZE,
        PIXELS_PER_TILE,
        endpoint=False,
    )

    lon_grid, lat_grid = np.meshgrid(x_coords, y_coords)

    return pd.DataFrame(
        {
            "lat": lat_grid.flatten(),
            "long": lon_grid.flatten(),
            "radiance": radiance.flatten(),
        }
    )


In [68]:
files = [
    ("VNP46A2.A2020001.h25v06.001.2021053030312.h5", 25, 6),
    ("VNP46A2.A2020001.h25v07.001.2021053030351.h5", 25, 7),
    ("VNP46A2.A2020001.h25v08.001.2021053030235.h5", 25, 8),
    ("VNP46A2.A2020001.h26v06.001.2021053030428.h5", 26, 6),
    ("VNP46A2.A2020001.h26v07.001.2021053031026.h5", 26, 7),
    ("VNP46A2.A2020001.h26v08.001.2021053030301.h5", 26, 8),
]

rad_df = pd.concat(
    [
        get_rad_df(f"../../data/spring_2025_data/nighttime-lights-raw/{f}", h, v)
        for f, h, v in files
    ],
    ignore_index=True,
)


In [69]:
all_joined = join_euclidean(
    rad_df, pop_density_joined, ["long", "lat"], ["Long", "Lat"], "radiance"
)

In [ ]:
all_joined = pd.read_csv(
    "../../data/spring_2025_data/cleaned_alldata_with_urbanization.csv"
)
all_joined["high_radiance"] = (
    all_joined["radiance"] >= all_joined["radiance"].quantile(0.9)
).astype(int)

In [5]:
all_joined.head()

,Unnamed: 0.1,Unnamed: 0,Date_of_observation,Species_name,Lat,Long,State_name,Leaves_fresh,Leaves_mature,Leaves_old,...,Fruits_ripe,Fruits_open,Year,Week,Species_id,UN_2020_DS,is_urban,pop_bin,radiance,high_radiance
0,0,1,2020-01-01,Indian Almond-Terminalia catappa,12.15386,75.22397,Kerala,2.0,0.0,0.0,...,-2.0,-2.0,2020.0,0,1085.0,643.461262,1,0,5,0
1,1,2,2020-01-01,Indian Almond-Terminalia catappa,12.15386,75.22397,Kerala,2.0,0.0,0.0,...,-2.0,-2.0,2020.0,0,1085.0,643.461262,1,0,5,0
2,2,3,2020-01-01,Fish-tail Palm-Caryota urens,12.14060,75.22145,Kerala,0.0,2.0,0.0,...,-2.0,-2.0,2020.0,0,1019.0,643.461262,1,0,4,0
3,3,4,2020-01-01,Mast Tree-Monoon longifolium,12.14060,75.22145,Kerala,1.0,2.0,0.0,...,-2.0,-2.0,2020.0,0,1065.0,643.461262,1,0,4,0
4,4,5,2020-01-01,Indian Almond-Terminalia catappa,12.14060,75.22145,Kerala,0.0,1.0,2.0,...,-2.0,-2.0,2020.0,0,1085.0,643.461262,1,0,4,0


In [7]:
all_joined.to_csv("../../data/spring_2025_data/cleaned_alldata_with_urbanization.csv")